# End-to-End Production Pipeline (Streamfix v2): Vanilla MATD3 / MASAC on NVIDIA Warp GPU + Real UavNetSim CSMA/CA (Kaggle T4x2 Public Account 1)
1. Bootstrap repo + install dependencies (`warp-lang`, `simpy`, `trimesh`, `wandb`, `huggingface_hub`, `pyyaml`) + auto-load Kaggle Secrets (`WANDB_API_KEY`, `HF_TOKEN`, `HF_DATASET_REPO`) with zero token leakage.
2. Run T4x2 GPU Acceptance Suite (`test_gpu_warp_parity_and_cudagraph_t4.py`): CPU-GPU parity, CUDA Graph stream-sync, replay non-aliasing, 1M FP16 replay VRAM check, and real UavNetSim CSMA/CA.
3. Run bounded CPU `--reference-smoke` sanity check.
4. Run resumable 12h-safe Production Training (`--mode production --resume --max-wall-clock-s 37800`) with incremental Hugging Face backups and continuous W&B Online logging.
5. 5-Seed Paper Orchestration (`[44, 45, 46, 47, 48]`) + `rliable` / `marl-eval` Seed-Level Aggregation.

In [ ]:
# Cell 1: Bootstrap Repo, Install Dependencies (incl. trimesh), Load Kaggle Secrets Safely
import os, pathlib, subprocess, sys

# 1) Locate source tree if mounted under /kaggle/input
cwd = pathlib.Path.cwd()
if not (cwd / "uav_network").exists():
    for cand in sorted(pathlib.Path("/kaggle/input").glob("**/uav_network")):
        os.chdir(cand.parent)
        break
print("Working repo:", pathlib.Path.cwd())
os.environ["PYTHONPATH"] = str(pathlib.Path.cwd())
os.environ.setdefault("WARP_CACHE_PATH", "/kaggle/working/.warp_cache")

# 2) Install required runtime packages (trimesh fixes v2 UavNetSim scene import error)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "warp-lang", "simpy", "trimesh", "wandb", "huggingface_hub", "pyyaml"
], check=True)

# 3) Auto-load Kaggle Secrets into os.environ (never printed, stays private on Public notebooks)
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    for _k in ("WANDB_API_KEY", "HF_TOKEN", "HF_DATASET_REPO"):
        if not os.environ.get(_k):
            try:
                os.environ[_k] = _sec.get_secret(_k)
            except Exception:
                pass
except ImportError:
    pass

import torch, warp as wp, simpy, trimesh
wp.init()
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)")
print("Warp:", wp.__version__, "| SimPy:", simpy.__version__, "| trimesh:", trimesh.__version__)
print("WANDB_API_KEY set:", bool(os.environ.get("WANDB_API_KEY")), "| HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# Cell 2: Full T4x2 GPU Acceptance Suite (Parity, Streamfix, Replay Non-Aliasing, 1M VRAM, Real UavNetSim)
import subprocess, sys
subprocess.run([sys.executable, "tests/test_gpu_warp_parity_and_cudagraph_t4.py"], check=True)

In [ ]:
# Cell 3: Fast Bounded CPU Reference Smoke Check
import shutil, subprocess, sys, pathlib
smoke_dir = pathlib.Path("/kaggle/working/smoke_matd3_ref")
if smoke_dir.exists():
    shutil.rmtree(smoke_dir)
subprocess.run([
    sys.executable, "-m", "uav_network.experiments.train_slotted_vanilla",
    "--config", "configs/vanilla_matd3_shared_2d_ds_aoi.yaml",
    "--output", str(smoke_dir),
    "--reference-smoke",
    "--total-steps", "256", "--num-envs", "4", "--eval-episodes", "2", "--max-steps", "25",
    "--wandb-mode", "disabled",
], check=True)

In [ ]:
# Cell 4: Resumable Kaggle-12h-Safe Production Run (Parameterize ALGO & SEED)
import os, pathlib, subprocess, sys

ALGO = os.environ.get("UAV_ALGO", "matd3")          # "matd3" or "masac"
SEED = int(os.environ.get("UAV_SEED", "44"))
MAX_WALL_S = os.environ.get("UAV_MAX_WALL_S", "37800")  # 10.5h graceful exit before Kaggle 12h limit
HF_REPO = os.environ.get("HF_DATASET_REPO", "")
WANDB_MODE = "online" if os.environ.get("WANDB_API_KEY") else "offline"

cfg_path = f"configs/vanilla_{ALGO}_shared_2d_ds_aoi.yaml"
out_dir = pathlib.Path(f"/kaggle/working/prod_{ALGO}_s{SEED}")
out_dir.mkdir(parents=True, exist_ok=True)

# Optional auto-pull prior checkpoint from Hugging Face Dataset for multi-session resume
RESUME_RUN_SUBPATH = os.environ.get("HF_RESUME_SUBPATH", "")
if HF_REPO and os.environ.get("HF_TOKEN") and RESUME_RUN_SUBPATH:
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=HF_REPO, repo_type="dataset", token=os.environ["HF_TOKEN"],
        allow_patterns=f"{RESUME_RUN_SUBPATH}/*", local_dir=str(out_dir.parent / "hf_pull"),
    )
    pulled = out_dir.parent / "hf_pull" / RESUME_RUN_SUBPATH
    if pulled.exists():
        import shutil
        shutil.copytree(pulled, out_dir, dirs_exist_ok=True)
        print("Pulled checkpoint from HF:", pulled)

cmd = [
    sys.executable, "-m", "uav_network.experiments.train_slotted_vanilla",
    "--config", cfg_path,
    "--output", str(out_dir),
    "--mode", "production",
    "--seed", str(SEED),
    "--max-wall-clock-s", str(MAX_WALL_S),
    "--wandb-mode", WANDB_MODE,
    "--wandb-project", "uav_search_target",
]
if (out_dir / "checkpoints" / "latest_checkpoint.pt").exists():
    cmd.append("--resume")
if HF_REPO:
    cmd.extend(["--hf-repo", HF_REPO])
print("Launching:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Cell 5: 5-Seed Paper Orchestration Template & rliable/marl-eval Aggregation
import json, pathlib, subprocess, sys

completed_runs = [str(p) for p in sorted(pathlib.Path("/kaggle/working").glob("prod_*_s*")) if (p / "marl_eval_results.json").exists()]
if completed_runs:
    agg_out = "/kaggle/working/five_seed_rliable_summary.json"
    subprocess.run([
        sys.executable, "-m", "uav_network.experiments.aggregate_five_seed_rliable",
        "--runs", *completed_runs, "--out", agg_out,
    ], check=True)
    print(pathlib.Path(agg_out).read_text(encoding="utf-8"))
else:
    print("No completed production runs found yet under /kaggle/working/prod_*_s*.")

In [ ]:
# Cell 6: Inspect Final Dual-Fidelity Report & SHA-256 Forensic Manifest
import json, pathlib
for p in sorted(pathlib.Path("/kaggle/working").glob("prod_*_s*/summary_metrics.json")):
    rep = json.loads(p.read_text(encoding="utf-8"))
    print("===", p.parent.name, "===")
    print("gpu_warp_verified:", rep.get("gpu_warp_verified"), "| vendor_uavnetsim_verified:", rep.get("vendor_uavnetsim_verified"))
    print("sim_to_net_gap:", json.dumps(rep.get("sim_to_net_gap", {}), indent=2))